# Professional Data Cleaning — Customer Dataset

**Objective:** turn deliberately inconsistent customer records into a clean, analysis-ready CSV and document each decision.

The input file is synthetic practice data. This notebook is designed to demonstrate a repeatable cleaning workflow, not to describe real customers.


## 1. Import libraries and load data

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path("../data/messy_customer_data.csv")
OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

df_raw = pd.read_csv(DATA_PATH)
df = df_raw.copy()
print("Shape:", df.shape)
display(df.head())
df.info()


## 2. Data quality report — before cleaning

In [ ]:
before_nulls = df.isna().sum().rename("NullCount").to_frame()
before_duplicates = int(df.duplicated().sum())
before_dtypes = df.dtypes.astype(str).rename("DataType").to_frame()
display(before_nulls)
print("Exact duplicate rows:", before_duplicates)
display(before_dtypes)
display(df.describe(include="all").T)


### Quality checks and issues found

The source includes missing values, exact duplicate records, inconsistent category spelling/case, mixed date formats, numeric values stored as text (including currency symbols), impossible values, and numeric outliers. The cells below make each correction explicit.

## 3. Remove exact duplicates

In [ ]:
duplicate_count = int(df.duplicated().sum())
df = df.drop_duplicates().copy()
print(f"Removed {duplicate_count} exact duplicate rows.")


## 4. Standardise text and categories

- Trim whitespace and uppercase customer IDs; IDs remain strings because they are labels, not quantities.
- Map `M`, `m`, `male` to `Male`; map female variants to `Female`.
- Map spelling/case variants such as `BANGALORE` and `bengaluru` to `Bengaluru`.
- Standardise emails to lowercase.

In [ ]:
df["CustomerID"] = df["CustomerID"].astype("string").str.strip().str.upper()
df["CustomerName"] = df["CustomerName"].astype("string").str.strip()
df["Email"] = df["Email"].astype("string").str.strip().str.lower()

gender_map = {"m":"Male", "male":"Male", "f":"Female", "female":"Female", "other":"Other"}
df["Gender"] = df["Gender"].astype("string").str.strip().str.lower().map(gender_map)

city_map = {"hyderabad":"Hyderabad", "bengaluru":"Bengaluru",
            "bangalore":"Bengaluru", "mumbai":"Mumbai", "chennai":"Chennai"}
df["City"] = df["City"].astype("string").str.strip().str.lower().map(city_map)


## 5. Correct dates and numeric data types

Dates are parsed into `datetime64[ns]`. Invalid date strings become `NaT`. Currency symbols and commas are removed before converting money columns to numeric values.

In [ ]:
df["SignupDate"] = pd.to_datetime(df["SignupDate"], format="mixed", errors="coerce", dayfirst=True)

def parse_numeric(series):
    cleaned = (series.astype("string")
               .str.replace(r"[^0-9.\\-]", "", regex=True)
               .replace("", pd.NA))
    return pd.to_numeric(cleaned, errors="coerce")

for col in ["Age", "AnnualIncome", "PurchaseAmount", "SatisfactionScore"]:
    df[col] = parse_numeric(df[col])

display(df.dtypes)


## 6. Missing-data decisions

| Column | Strategy | Reason |
|---|---|---|
| CustomerID | Drop row if missing | Required to identify a customer; do not invent IDs |
| SignupDate | Drop row if invalid/missing | Essential for time-based analysis |
| Age | Median | Robust to skew and extreme ages |
| AnnualIncome | Median | Robust to skew and high incomes |
| PurchaseAmount | Median | Robust to skew and large purchases |
| SatisfactionScore | Median, then round | Ordinal 1–5 score; median is less sensitive to extremes |
| Gender | Mode | Small number of categorical labels |
| City | Mode | Standardised location category |
| CustomerName | `Unknown` placeholder | Preserve row without inventing a name |
| Email | `Not Provided` placeholder | Preserve row; do not fabricate contact details |

Numeric values outside plausible/domain ranges are marked missing before imputation.

In [ ]:
df = df.dropna(subset=["CustomerID", "SignupDate"]).copy()
df.loc[~df["Age"].between(0, 110), "Age"] = np.nan
df.loc[df["AnnualIncome"] < 0, "AnnualIncome"] = np.nan
df.loc[df["PurchaseAmount"] < 0, "PurchaseAmount"] = np.nan
df.loc[~df["SatisfactionScore"].between(1, 5), "SatisfactionScore"] = np.nan

for col in ["Age", "AnnualIncome", "PurchaseAmount", "SatisfactionScore"]:
    df[col] = df[col].fillna(df[col].median())
for col in ["Gender", "City"]:
    mode = df[col].mode(dropna=True)
    df[col] = df[col].fillna(mode.iloc[0] if not mode.empty else "Unknown")
df["CustomerName"] = df["CustomerName"].fillna("Unknown").replace("", "Unknown")
df["Email"] = df["Email"].fillna("Not Provided").replace("", "Not Provided")


## 7. Outlier detection using IQR

For each numeric field, calculate Q1, Q3 and IQR. Values outside `Q1 − 1.5 × IQR` and `Q3 + 1.5 × IQR` are flagged. We cap these values at the IQR fences instead of deleting rows: a high income or purchase can be legitimate and deleting the whole customer would lose useful information.

In [ ]:
outlier_rows = []
for col in ["Age", "AnnualIncome", "PurchaseAmount", "SatisfactionScore"]:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
    count = int(((df[col] < lower) | (df[col] > upper)).sum())
    outlier_rows.append({"Column": col, "OutlierCount": count,
                         "LowerFence": lower, "UpperFence": upper,
                         "Decision": "Cap; retain rows"})
    df[col] = df[col].clip(lower, upper)
outlier_report = pd.DataFrame(outlier_rows)
display(outlier_report)


## 8. Enforce final data types and validate

In [ ]:
for col in ["CustomerID", "CustomerName", "Gender", "City", "Email"]:
    df[col] = df[col].astype("string")
df["SignupDate"] = pd.to_datetime(df["SignupDate"])
df["Age"] = df["Age"].round().astype("Int64")
df["SatisfactionScore"] = df["SatisfactionScore"].round().clip(1,5).astype("Int64")
df["AnnualIncome"] = df["AnnualIncome"].astype("float64")
df["PurchaseAmount"] = df["PurchaseAmount"].astype("float64")
df = df.sort_values("CustomerID").reset_index(drop=True)

display(df.head())
display(df.dtypes)
print("Remaining null cells:", int(df.isna().sum().sum()))
print("Remaining exact duplicates:", int(df.duplicated().sum()))


## 9. Before vs. after summary

In [ ]:
expected = {
    "CustomerID":"string", "CustomerName":"string", "Gender":"string",
    "Age":"Int64", "SignupDate":"datetime64[ns]", "City":"string",
    "Email":"string", "AnnualIncome":"float64", "PurchaseAmount":"float64",
    "SatisfactionScore":"Int64"
}
def dtype_accuracy(frame):
    good = sum(str(frame[c].dtype) == t for c,t in expected.items())
    return round(100*good/len(expected), 1)

summary = pd.DataFrame([
    {"Stage":"Before cleaning", "Rows":len(df_raw), "TotalNullCells":int(df_raw.isna().sum().sum()),
     "DuplicateRows":int(df_raw.duplicated().sum()), "DtypeAccuracyPercent":dtype_accuracy(df_raw)},
    {"Stage":"After cleaning", "Rows":len(df), "TotalNullCells":int(df.isna().sum().sum()),
     "DuplicateRows":int(df.duplicated().sum()), "DtypeAccuracyPercent":dtype_accuracy(df)}
])
display(summary)


## 10. Export clean dataset and audit reports

In [ ]:
df.to_csv(OUTPUT_DIR / "cleaned_customer_data.csv", index=False, date_format="%Y-%m-%d")
before_nulls.to_csv(OUTPUT_DIR / "nulls_by_column_before.csv")
outlier_report.to_csv(OUTPUT_DIR / "outlier_report.csv", index=False)
summary.to_csv(OUTPUT_DIR / "data_quality_before_after.csv", index=False)
print("Saved cleaned CSV and reports in:", OUTPUT_DIR.resolve())
